# 09 — Dot Summaries

## Approach: LLM-Generated Bulgarian Summaries

Each dot is summarised in 1–2 Bulgarian sentences using **BgGPT** running locally via Ollama.
The input is the titles of all articles in the dot — no full text included.

Titles alone are sufficient because Bulgarian news headlines are descriptive and carry the
key facts (names, places, actions).

**Singleton dots (size=1)** are not sent to BgGPT — the single article's title is used
directly as the summary. 77% of dots are singletons, so this reduces the LLM calls
from 20,974 to ~4,820, cutting estimated runtime from ~10h to ~2.3h.

## Result files
- `data/processed/dot_summaries.pkl` — dict `{(story_id, dot_idx): summary_string}`

In [ ]:
import pickle
from pathlib import Path

import ollama
import pandas as pd

REPO_ROOT = Path("../").resolve()
MODEL = "bggpt:latest"
CHECKPOINT_PATH = REPO_ROOT / "data/processed/dot_summaries.pkl"

# Load Data

In [6]:
df = pd.read_parquet(REPO_ROOT / "data/processed/articles_clean.parquet")

with open(REPO_ROOT / "data/processed/dots_louvain.pkl", "rb") as f:
    all_dots = pickle.load(f)

total_dots = sum(len(dots) for dots in all_dots.values())
print(f"Stories : {len(all_dots):,}")
print(f"Total dots: {total_dots:,}")

Stories : 18,473
Total dots: 20,974


# Prompt & Generate Function

In [7]:
def build_prompt(dot, df):
    titles = "\n".join([f"- {df.iloc[i]['title']}" for i in dot["indices"]])
    return (
        "Ти си редактор на новинарски сайт. По-долу са заглавията на няколко новини "
        "за едно и също събитие, публикувани в един и същи ден от различни източници. "
        "Напиши 1-2 изречения на български, които обобщават какво се е случило. "
        "Бъди конкретен — включи имена, места и действия. "
        "Не добавяй въведение или обяснение, само резюмето.\n\n"
        f"Новини:\n{titles}"
    )


def generate_summary(dot, df, model=MODEL):
    if dot["size"] == 1:
        return df.iloc[dot["indices"][0]]["title"]
    prompt = build_prompt(dot, df)
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"].strip()

# Speed Test

Run on 20 dots to estimate total time before committing to the full dataset.

In [10]:
# test on 5 multi-article dots
examples = [
    (sid, idx, dot)
    for sid, dots in all_dots.items()
    for idx, dot in enumerate(dots)
    if dot["size"] > 1
][:5]

In [9]:
for sid, idx, dot in examples:
    summary = generate_summary(dot, df)
    print(
        f"Story {sid} | Dot {idx} | {dot['effective_start'].strftime('%d %b %H:%M')} | {dot['size']} articles"
    )
    print(f"Summary: {summary}")
    print("Articles:")
    for i in dot["indices"]:
        row = df.iloc[i]
        print(f"  - [{row['source']}] {row['title']}")
    print()

Story 3084 | Dot 0 | 24 Apr 21:16 | 3 articles
Summary: САЩ наложиха нови санкции срещу китайската рафинерия „Синопек“, която е сред основните клиенти на ирански петрол.
Articles:
  - [actualno] Заради Иран: САЩ въведоха санкции срещу китайска петролна рафинерия
  - [24chasa] САЩ наложиха санкции на китайска петролна рафинерия заради връзки с Техеран
  - [fakti] САЩ наложиха санкции на китайска рафинерия, основен купувач на петрол от Иран

Story 8 | Dot 0 | 24 Apr 21:45 | 2 articles
Summary: В неделя вечер „Реал“ Мадрид допусна изненадваща загуба с 0:1 от „Бетис“, а попадението за домакините падна в добавеното време. Това е трета поредна загуба за тима на Карло Анчелоти, който вече има само една победа в последните шест мача.
Articles:
  - [actualno] Реал Мадрид пак катастрофира! Гол в последните секунди стъжни "кралете"
  - [24chasa] "Реал" прецакан в 94-ата мин от "Бетис", в последните 6 мача има само 1 победа

Story 8 | Dot 1 | 10 May 04:40 | 2 articles
Summary: В дербито между Реал